In [ ]:
# Import necessary libraries
import os
import sys
# Add the utils folder to the system path so we can import custom modules
sys.path.append(os.path.abspath(os.path.join("..")))
import matplotlib.pyplot as plt
import torch
import numpy as np
import torch.nn as nn
import pandas as pd
from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data, load_test_data, feature_names

# Set the plotting style for better-looking visualizations
plt.style.available
plt.style.use('seaborn-v0_8')

In [ ]:
seed_everything(42)  # for reproducibility

In [ ]:
print("Loading train data...")
train_dataloader, val_dataloader, pipeline = load_data(
    train_path = "datasets_challenge/train",  # Path to training data
    train_ratio = 0.8,  # Use 80% for training, 20% for validation
    batchsize = 512,  # Number of samples per batch
    columns_to_standardize = feature_names,  # Option to specify which columns to standardize
    columns_to_drop = ["sideSlip", "vyCG", "time", "longitude","latitude"],  # Remove these from features
    sequence_length = 20,  # Number of time steps in each sequence (t-19 to t0)
    threshold = True,  # Enable velocity thresholding
    threshold_value = 3.0,  # Only keep samples where vehicle speed > 3 m/s
    threshold_column = "vxCG",  # Column to apply threshold on (longitudinal velocity)
    target = "sideSlip",  # The variable we want to predict
    scaling = True,  # Apply standardization to features
    seed = 42,  # Random seed for train/val split
)

In [ ]:
# Get one batch of data from the training dataloader
X, y = next(iter(train_dataloader))

# Visualize the first sample in the batch
plt.figure(figsize=(8,4))
plt.plot(X[0,:,:], label=feature_names)  # Plot all 22 features over 20 time steps
plt.title("Sideslip: "+str(round(y[0].item(),2))+"°")  # Show the target value
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
plt.tight_layout()
plt.xlabel("time step")
plt.ylabel("standardized feature value")
plt.show()

In [ ]:
num_features = len(feature_names)
print("Num features:", num_features)

In [ ]:
import torch
import torch.nn as nn

class GRURegressor(nn.Module):
    def __init__(self,
                 input_size,
                 hidden_size=128,
                 num_layers=2,
                 fc_units=64,
                 dropout=0.4,
                 activation_fct=nn.ReLU()):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_size, fc_units),
            activation_fct,
            nn.Dropout(dropout),
            nn.Linear(fc_units, 1)
        )

    def forward(self, x):
        # x: (batch, seq_len, features)
        out, _ = self.gru(x)             # out: (batch, seq_len, hidden)
        last_timestep = out[:, -1, :]    # use last time step
        y_hat = self.fc(last_timestep)
        return y_hat
    
    
def band_weighted_l1(pred, target):
    err = torch.abs(pred - target)
    w = torch.ones_like(target)

    # many-shot (|angle| ≤ 1°)
    many_mask = torch.abs(target) <= 1.0
    # medium-shot (1–5°)
    medium_mask = (torch.abs(target) > 1.0) & (torch.abs(target) <= 5.0)
    # few-shot (|angle| > 5°)
    few_mask = torch.abs(target) > 5.0

    w[many_mask] = 1.0
    w[medium_mask] = 2.0
    w[few_mask] = 6.0

    return (w * err).mean()
